# M3 House Resale Price Model

This notebook turns the project placeholder into a complete supervised learning workflow for resale price prediction.

Expected files in `../data/`:
- `train.csv`: labeled training data that includes the target column `resale_price`
- `test.csv`: unlabeled scoring data that includes the identifier column `Id`

Generated file:
- `../output/Team_10_submission.csv`

If `train.csv` is missing or empty, the notebook stops early with a clear message instead of producing a misleading model.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 200)

def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "m3-project", cwd.parent]

    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "workflow").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the m3-project root. Run the notebook from the project workspace."
    )


PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
OUTPUT_PATH = OUTPUT_DIR / "Team_10_submission.csv"

TARGET_COLUMN = "resale_price"
ID_CANDIDATES = ["Id", "id"]


In [ ]:
def load_required_csv(path: Path, label: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"{label} file not found at {path}. Add the expected CSV before running the notebook."
        )

    try:
        df = pd.read_csv(path, low_memory=False)
    except pd.errors.EmptyDataError as exc:
        raise ValueError(
            f"{label} file at {path} is empty. Add real data before training the model."
        ) from exc

    if df.empty:
        raise ValueError(f"{label} file at {path} has no rows.")

    return df


train_df = load_required_csv(TRAIN_PATH, "Training")
test_df = load_required_csv(TEST_PATH, "Test")

ID_COLUMN = next((column for column in ID_CANDIDATES if column in test_df.columns), None)

if TARGET_COLUMN not in train_df.columns:
    raise KeyError(
        f"Training data must include the target column '{TARGET_COLUMN}'. "
        f"Available columns: {list(train_df.columns)}"
    )

if ID_COLUMN is None:
    raise KeyError(
        "Test data must include one of the identifier columns "
        f"{ID_CANDIDATES}. "
        f"Available columns: {list(test_df.columns)}"
    )

display(train_df.head())
display(test_df.head())

print(f"train shape: {train_df.shape}")
print(f"test shape: {test_df.shape}")


In [ ]:
X = train_df.drop(columns=[TARGET_COLUMN]).copy()
y = train_df[TARGET_COLUMN].copy()

if ID_COLUMN in X.columns:
    X = X.drop(columns=[ID_COLUMN])

X_test = test_df.copy()
test_ids = X_test[ID_COLUMN]

if ID_COLUMN in X_test.columns:
    X_test = X_test.drop(columns=[ID_COLUMN])

missing_test_columns = sorted(set(X.columns) - set(X_test.columns))
extra_test_columns = sorted(set(X_test.columns) - set(X.columns))

if missing_test_columns:
    raise ValueError(
        "Test data is missing training feature columns: " + ", ".join(missing_test_columns)
    )

if extra_test_columns:
    print("Dropping test-only columns:", extra_test_columns)
    X_test = X_test.drop(columns=extra_test_columns)

X_test = X_test[X.columns]

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

for column in categorical_features:
    X[column] = X[column].astype("string")
    X_test[column] = X_test[column].astype("string")

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ],
    remainder="drop",
)

print(f"numeric features: {len(numeric_features)}")
print(f"categorical features: {len(categorical_features)}")


In [ ]:
candidate_models = {
    "random_forest": RandomForestRegressor(
        n_estimators=120,
        max_depth=30,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    ),
    "extra_trees": ExtraTreesRegressor(
        n_estimators=160,
        max_depth=30,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    ),
}

model_scores = []

for model_name, estimator in candidate_models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator),
    ])
    rmse_scores = -cross_val_score(
        pipeline,
        X,
        y,
        cv=3,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )
    model_scores.append(
        {
            "model": model_name,
            "cv_rmse_mean": rmse_scores.mean(),
            "cv_rmse_std": rmse_scores.std(),
        }
    )

scores_df = pd.DataFrame(model_scores).sort_values("cv_rmse_mean")
display(scores_df)

best_model_name = scores_df.iloc[0]["model"]
best_estimator = candidate_models[best_model_name]
print(f"Selected model: {best_model_name}")


In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

validation_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_estimator),
])

validation_pipeline.fit(X_train, y_train)
valid_predictions = validation_pipeline.predict(X_valid)

rmse = mean_squared_error(y_valid, valid_predictions, squared=False)
mae = mean_absolute_error(y_valid, valid_predictions)

print(f"Validation RMSE: {rmse:,.2f}")
print(f"Validation MAE: {mae:,.2f}")


In [ ]:
final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_estimator),
])

final_pipeline.fit(X, y)
test_predictions = final_pipeline.predict(X_test)

submission_df = pd.DataFrame(
    {
        ID_COLUMN: test_ids,
        "Predicted": np.round(test_predictions, 2),
    }
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
submission_df.to_csv(OUTPUT_PATH, index=False)

display(submission_df.head())
print(f"Submission saved to: {OUTPUT_PATH}")


## Recommended Next Step

Run the notebook after adding the real `train.csv` file. If the target column name differs from `resale_price`, update `TARGET_COLUMN` in the configuration cell before training.